In [1]:
# Cell 1: Check libraries and initialize Earth Engine
import ee
import numpy as np
import pandas as pd

# Initialize Earth Engine using your newly configured Cloud Project
try:
    ee.Initialize(project="marine-monitoring-508504")
    print("Google Earth Engine successfully initialized!")
except Exception as e:
    print("Initialization failed. Run ee.Authenticate() first:", e)

Google Earth Engine successfully initialized!


In [2]:
# Cell 2: Define target Marine Parks
MARINE_PARKS = {
    "Pulau Redang": {"lat": 5.7878, "lon": 103.0164},
    "Pulau Perhentian": {"lat": 5.9145, "lon": 102.7397},
    "Pulau Tioman": {"lat": 2.8000, "lon": 104.1833},
    "Pulau Payar": {"lat": 6.0633, "lon": 100.0414},
    "Pulau Sipadan": {"lat": 4.1147, "lon": 118.6287},
}

In [3]:
# Cell 3: Satellite Data Fetching Pipeline
def fetch_marine_data(start_year=2023, end_year=2026):
    records = []

    for name, coords in MARINE_PARKS.items():
        # Buffer marine point by 2km to average surrounding ocean water
        point = ee.Geometry.Point([coords["lon"], coords["lat"]])
        region = point.buffer(2000)

        for year in range(start_year, end_year + 1):
            for month in range(1, 13):
                start_date = f"{year}-{month:02d}-01"
                end_date = (
                    f"{year+1}-01-01" if month == 12 else f"{year}-{month+1:02d}-01"
                )

                # 1. Fetch NOAA SST (Daily Optimum Interpolation)
                sst_col = (
                    ee.ImageCollection("NOAA/CDR/OISST/V2_1")
                    .filterDate(start_date, end_date)
                    .select("sst")
                )

                sst_img = sst_col.mean()
                sst_val = (
                    sst_img.reduceRegion(
                        reducer=ee.Reducer.mean(), geometry=region, scale=5000
                    ).get("sst")
                )

                # 2. Fetch Sentinel-2 Reflectance (Turbidity Proxy - Red Band B4)
                s2_col = (
                    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                    .filterDate(start_date, end_date)
                    .filterBounds(region)
                    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
                    .select("B4")
                )

                s2_img = s2_col.mean()
                turb_val = (
                    s2_img.reduceRegion(
                        reducer=ee.Reducer.mean(), geometry=region, scale=10
                    ).get("B4")
                )

                # Convert Server Objects to Floats safely
                try:
                    # NOAA OISST raw values are scaled by 0.01
                    sst = (
                        round(float(sst_val.getInfo()) * 0.01, 2)
                        if sst_val.getInfo() is not None
                        else None
                    )
                except Exception:
                    sst = None

                try:
                    turbidity = (
                        round(float(turb_val.getInfo()), 4)
                        if turb_val.getInfo() is not None
                        else None
                    )
                except Exception:
                    turbidity = None

                month_str = f"{year}-{month:02d}"
                records.append(
                    {
                        "marine_park": name,
                        "month": month_str,
                        "sst_celsius": sst,
                        "turbidity_index": turbidity,
                    }
                )
                print(
                    f"Fetched {name} [{month_str}] -> SST: {sst}°C | Turbidity: {turbidity}"
                )

    df = pd.DataFrame(records)
    df.to_csv("raw_marine_environmental_data.csv", index=False)
    print("\nExtraction Complete! Saved to raw_marine_environmental_data.csv")
    return df


# Execute extraction
df_env = fetch_marine_data(2023, 2026)

Fetched Pulau Redang [2023-01] -> SST: 27.81°C | Turbidity: None
Fetched Pulau Redang [2023-02] -> SST: 27.58°C | Turbidity: None
Fetched Pulau Redang [2023-03] -> SST: 27.83°C | Turbidity: 2749.5751
Fetched Pulau Redang [2023-04] -> SST: 30.03°C | Turbidity: 4397.4225
Fetched Pulau Redang [2023-05] -> SST: 30.78°C | Turbidity: 977.1613
Fetched Pulau Redang [2023-06] -> SST: 30.09°C | Turbidity: 377.2795
Fetched Pulau Redang [2023-07] -> SST: 30.11°C | Turbidity: 1576.6289
Fetched Pulau Redang [2023-08] -> SST: 30.02°C | Turbidity: 899.671
Fetched Pulau Redang [2023-09] -> SST: 29.95°C | Turbidity: None
Fetched Pulau Redang [2023-10] -> SST: 30.32°C | Turbidity: 1950.6053
Fetched Pulau Redang [2023-11] -> SST: 29.86°C | Turbidity: None
Fetched Pulau Redang [2023-12] -> SST: 28.36°C | Turbidity: None
Fetched Pulau Redang [2024-01] -> SST: 28.0°C | Turbidity: None
Fetched Pulau Redang [2024-02] -> SST: 28.55°C | Turbidity: 926.4059
Fetched Pulau Redang [2024-03] -> SST: 29.26°C | Turbidi

In [7]:
# Cell 4 (REPLACED): Visitor Data Proxy from Real Review Volume
import pandas as pd

# Load the cleaned marine reviews (from Person 4's work)
df = pd.read_csv("sample_data/clean_reviews_marine.csv")

# Map each resort's specific name back to its generic park name,
# so it matches the environmental data's park names
RESORT_TO_PARK = {
    "The Taaras Beach & Spa Resort": "Pulau Redang",
    "Perhentian Island Resort": "Pulau Perhentian",
    "Berjaya Tioman Resort - Malaysia": "Pulau Tioman",
    "Sipadan Kapalai Dive Resort": "Pulau Sipadan",
    "Pulau Payar Marine Park": "Pulau Payar",
}

df["marine_park"] = df["destination"].map(RESORT_TO_PARK)
df = df.dropna(subset=["marine_park"])

# Count how many reviews were posted per park per month —
# this becomes our visitor volume proxy
monthly_counts = (
    df.groupby(["marine_park", "month"])
    .size()
    .reset_index(name="visitors")
)

monthly_counts.to_csv("raw_marine_visitor_records.csv", index=False)
print(f"Saved {len(monthly_counts)} rows -> raw_marine_visitor_records.csv")

Saved 102 rows -> raw_marine_visitor_records.csv


In [8]:
# Cell 5: Run Cleaning Pipeline and Output Final Deliverables
import pandas as pd


def clean_marine_pipeline():
    
    # 1. Clean Environmental Data
    try:
        df_env = pd.read_csv("raw_marine_environmental_data.csv")
        before_env = len(df_env)

        # Standardize park names
        df_env["marine_park"] = df_env["marine_park"].str.strip().str.title()

        # Remove impossible temperature values (sensor glitches under 20°C or over 35°C)
        df_env = df_env[
            (df_env["sst_celsius"] >= 20.0) & (df_env["sst_celsius"] <= 35.0)
        ]

        # Fill missing turbidity gaps caused by cloudy satellite pixels (forward fill then back fill per park)
        df_env = df_env.sort_values(by=["marine_park", "month"])
        
        df_env["turbidity_index"] = df_env.groupby("marine_park")["turbidity_index"].transform(
        lambda g: g.interpolate(method="linear").ffill().bfill()
        )

        # Drop duplicate entries if any
        df_env = df_env.drop_duplicates(subset=["marine_park", "month"])

        # Export Deliverable #1
        df_env.to_csv("clean_marine_environmental_data.csv", index=False)
        print(
            f"Deliverable 1 Success! clean_marine_environmental_data.csv ({len(df_env)} rows saved)"
        )

    except FileNotFoundError:
        print(
            "Error: 'raw_marine_environmental_data.csv' missing. Run Step 1 first."
        )

    # 2. Clean Visitor Data
    
    try:
        df_vis = pd.read_csv("raw_marine_visitor_records.csv")

        # Standardize names and ensure numeric types
        df_vis["marine_park"] = df_vis["marine_park"].str.strip().str.title()
        df_vis["visitors"] = pd.to_numeric(df_vis["visitors"], errors="coerce")
        df_vis = df_vis.dropna(subset=["visitors"]).astype({"visitors": int})
        df_vis = df_vis.drop_duplicates(subset=["marine_park", "month"])

        # Export Deliverable #2
        df_vis.to_csv("clean_marine_visitor_arrivals.csv", index=False)
        print(
            f"Deliverable 2 Success! clean_marine_visitor_arrivals.csv ({len(df_vis)} rows saved)"
        )

    except FileNotFoundError:
        print(
            "Error: 'raw_marine_visitor_records.csv' missing. Run Step 4 first."
        )


# Run the function
clean_marine_pipeline()

Deliverable 1 Success! clean_marine_environmental_data.csv (225 rows saved)
Deliverable 2 Success! clean_marine_visitor_arrivals.csv (102 rows saved)


In [9]:
#missing values in turbidity will be filled with adjacent data

In [10]:
#verification
import pandas as pd
df = pd.read_csv("clean_marine_environmental_data.csv")
print("Min month:", df["month"].min())
print("Max month:", df["month"].max())

Min month: 2023-01
Max month: 2026-09
